<a href="https://colab.research.google.com/github/EMej34/das172-examen2-Edwin-Reyes./blob/main/Modulo_de_funciones_requeridas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Modulo de funciones requeridas.py

"""

from typing import List, Tuple, Dict, Any


# ---------------------------------------------------------------------------
# 1. Módulo de Validación y Coherencia Dimensional
# ---------------------------------------------------------------------------
def validar_matrices(cargas: List[List[float]],
                      capacidades: List[List[float]]) -> bool:

    if not isinstance(cargas, list) or not isinstance(capacidades, list):
        return False

    n_cargas = len(cargas)
    n_capacidades = len(capacidades)

    # Dimensión mínima N >= 2 y coincidencia de número de filas
    if n_cargas < 2 or n_capacidades < 2 or n_cargas != n_capacidades:
        return False

    # Todas las filas de 'cargas' deben existir y tener la misma longitud
    if any(not isinstance(fila, list) for fila in cargas):
        return False
    if any(not isinstance(fila, list) for fila in capacidades):
        return False

    m_cargas = len(cargas[0])
    m_capacidades = len(capacidades[0])

    # Dimensión mínima M >= 2 y coincidencia de columnas
    if m_cargas < 2 or m_capacidades < 2 or m_cargas != m_capacidades:
        return False

    # Regularidad: todas las filas de cada matriz miden lo mismo
    if any(len(fila) != m_cargas for fila in cargas):
        return False
    if any(len(fila) != m_capacidades for fila in capacidades):
        return False

    # Validación de rangos de valores
    for fila in cargas:
        for peso in fila:
            if not isinstance(peso, (int, float)) or isinstance(peso, bool):
                return False
            if peso < 0:
                return False

    for fila in capacidades:
        for capacidad in fila:
            if not isinstance(capacidad, (int, float)) or isinstance(capacidad, bool):
                return False
            if capacidad <= 0:
                return False

    return True


# ---------------------------------------------------------------------------
# 2. Módulo de Cálculo de Ocupación y Detección de Sobrecarga
# ---------------------------------------------------------------------------
def calcular_ocupacion_sobrecarga(cargas: List[List[float]],
                                   capacidades: List[List[float]]
                                   ) -> Dict[str, Any]:

    n = len(cargas)
    m = len(cargas[0])

    matriz_porcentajes: List[List[float]] = []
    celdas_sobrecargadas: List[Tuple[int, int]] = []

    for i in range(n):
        fila_porcentajes = []
        for j in range(m):
            porcentaje = (cargas[i][j] / capacidades[i][j]) * 100.0
            fila_porcentajes.append(porcentaje)
            if porcentaje > 100.0:
                celdas_sobrecargadas.append((i, j))
        matriz_porcentajes.append(fila_porcentajes)

    return {
        "matriz_porcentajes": matriz_porcentajes,
        "celdas_sobrecargadas": celdas_sobrecargadas,
    }


# ---------------------------------------------------------------------------
# 3. Módulo de Evaluación de Balance y Simetría
# ---------------------------------------------------------------------------
def evaluar_balance(cargas: List[List[float]],
                     tolerancia_kg: float) -> Dict[str, Any]:

    n = len(cargas)
    m = len(cargas[0])

    pesos_fila = [sum(fila) for fila in cargas]

    mitad = m // 2
    if m % 2 == 0:
        columnas_izquierda = range(0, mitad)
        columnas_derecha = range(mitad, m)
    else:
        # columna central (índice 'mitad') se omite por ser el eje de simetría
        columnas_izquierda = range(0, mitad)
        columnas_derecha = range(mitad + 1, m)

    suma_izquierda = sum(cargas[i][j] for i in range(n) for j in columnas_izquierda)
    suma_derecha = sum(cargas[i][j] for i in range(n) for j in columnas_derecha)

    desbalance_lateral = abs(suma_izquierda - suma_derecha)
    balance_ok = desbalance_lateral <= tolerancia_kg

    return {
        "pesos_fila": pesos_fila,
        "desbalance_lateral": desbalance_lateral,
        "balance_ok": balance_ok,
    }


# ---------------------------------------------------------------------------
# 4. Módulo de Extracción de Submatriz de Sobrecarga Crítica
# ---------------------------------------------------------------------------
def extraer_submatriz_critica(matriz_porcentajes: List[List[float]],
                               k: int, p: int) -> List[List[float]]:

    n = len(matriz_porcentajes)
    m = len(matriz_porcentajes[0]) if n > 0 else 0

    if k < 1 or p < 1 or k > n or p > m:
        raise ValueError(
            f"Ventana {k}x{p} inválida para una matriz de {n}x{m}."
        )

    mejor_promedio = None
    mejor_conteo_sobrecarga = -1
    mejor_submatriz: List[List[float]] = []

    for i in range(n - k + 1):
        for j in range(m - p + 1):
            # Copia (no referencia) de la ventana actual
            submatriz = [fila[j:j + p] for fila in matriz_porcentajes[i:i + k]]
            valores = [valor for fila in submatriz for valor in fila]

            promedio = sum(valores) / len(valores)
            conteo_sobrecarga = sum(1 for v in valores if v > 100.0)

            es_mejor = (
                mejor_promedio is None
                or promedio > mejor_promedio
                or (promedio == mejor_promedio
                    and conteo_sobrecarga > mejor_conteo_sobrecarga)
            )

            if es_mejor:
                mejor_promedio = promedio
                mejor_conteo_sobrecarga = conteo_sobrecarga
                mejor_submatriz = submatriz

    return mejor_submatriz